<a href="https://colab.research.google.com/github/spirosChv/neuro208/blob/main/Intro_to_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Intro to Neural Modeling in Python
This notebook covers the transition from basic Python logic to neural models.

# Step 0: What are Variables?
In Python, a **Variable** is a name that holds a value. Think of it as a labeled box. In neural modeling, we use variables to store our **inputs**, **weights**, and **biases**.

### Types of data we will use:
1. **Integers / Floats**: Single numbers (e.g., `weight = 0.5`).
2. **Strings**: Text inside quotes (e.g., `neuron_name = "Perceptron"`).
3. **Booleans**: True or False values (e.g., `is_active = True`).

In [ ]:
# Assigning Variables

# Let's define a single input and its weight
input_signal = 1.0  # This is a 'float' (decimal number)
weight = 0.8  # This is a 'float'
neuron_label = "Sensory Neuron" # This is a 'string' (text)

# We can perform math using these variable names
activation = input_signal * weight

# The print() function lets us see what is inside the variable
print(f"The name of the neuron is: {neuron_label}")
print(f"The calculated activation is: {activation}")

# Variables can be updated
activation = activation + 0.1
print(f"Updated activation: {activation}")

## 1. Python Fundamentals: Lists and Loops
To model a neuron, we need to handle multiple inputs. We use **Lists** to store data and **For Loops** to process it.

* **Lists `[]`**: Collections of data (like input signals).
* **For Loops**: A way to "walk" through a list and perform math on each item.

In [ ]:
# Our inputs (sensory signals) and weights (connection strength)
inputs = [1.0, 0.5, 0.2]
weights = [0.4, -0.2, 0.8]

# A loop to calculate the 'Weighted Sum' manually
weighted_sum = 0
for i in range(len(inputs)):
  contribution = inputs[i] * weights[i]
  weighted_sum += contribution
  print(f"Input {i} contributed {contribution:.2f}")

print(f"Total Weighted Sum: {weighted_sum:.2f}")

## 2. The McCulloch-Pitts (M-P) Neuron
The M-P neuron is a binary model. It treats inputs as 0 or 1. If the sum of inputs reaches a **Threshold**, the neuron fires (1).



### Concepts: Functions and If-Statements
* **Functions `def`**: A reusable block of code (our "Neuron").
* **If-Statements**: The logic that decides if the threshold is met.

In [ ]:
def mc_pitts_neuron(inputs, threshold):
  # Sum the binary inputs
  total = 0
  for x in inputs:
    total += x

  # Threshold Logic (Activation)
  if total >= threshold:
    return 1  # Fire!
  else:
    return 0  # Stay silent


# Test: Logic Gate (AND)
# Should only fire if both inputs are 1
print("M-P Output (AND gate):", mc_pitts_neuron([1, 1], 2))

## 3. The Perceptron (Rate-Based Model)
The Perceptron is more realistic. It uses continuous inputs (firing rates) and a **Bias** ($b$) to control the firing threshold.

The formula is:
$$z = \sum_{i=1}^N (x \cdot w) + b$$

where we assume $N$ inputs.

In [ ]:
def perceptron(inputs, weights, bias):
    # Weighted Sum
    z = 0
    for i in range(len(inputs)):
      z += inputs[i] * weights[i]

    # Add Bias
    z += bias

    # Activation (Step Function)
    # If z is 0 or positive, return 1, else 0
    if z >= 0:
      return 1
    else:
      return 0


# Test the Perceptron
sample_inputs = [0.7, 0.2]
sample_weights = [1.0, -0.5]
sample_bias = -0.3

result = perceptron(sample_inputs, sample_weights, sample_bias)
print(f"Perceptron Output: {result}")

## 4. The Learning Rule (Training the Neuron)
How does a neuron "learn"? If it predicts a `0` but the correct answer was `1`, it needs to increase its weights. If it predicts a `1` but the answer was `0`, it decreases them.

The formula for updating a weight is:
$$w_{new} = w_{old} + \eta \cdot (target - prediction) \cdot x$$

* **$\eta$ (Learning Rate)**: How big of a step we take (usually a small number like 0.1).
* **Target**: The actual correct answer
* **Prediction**: What the neuron guessed
* **Error**: $(target - prediction)$

In [ ]:
# Setup our Initial Parameters
inputs = [1, 0]  # Example: [Is it raining?, Have an umbrella?]
target = 1  # Goal: We want the neuron to say "Go outside" (1)
weights = [0.1, 0.1]
bias = -0.5
learning_rate = 0.1

def train_step(inputs, weights, bias, target, lr):
  # Get current prediction
  weighted_sum = sum(x * w for x, w in zip(inputs, weights)) + bias
  prediction = 1 if weighted_sum >= 0 else 0

  # Calculate Error
  error = target - prediction

  # Update weights and bias only if there was an error
  if error != 0:
    for i in range(len(weights)):
      weights[i] = weights[i] + (lr * error * inputs[i])
    bias = bias + (lr * error)
    print(f"Error detected! New weights: {weights}, New bias: {bias}")
  else:
    print("Prediction was correct. No changes made.")

  return weights, bias


# Run one training step
weights, bias = train_step(inputs, weights, bias, target, learning_rate)

## 5. Rate-Based Models: Integrating Over Time
In computational neuroscience, we often look at **Rate-Based Models** where the neuron's activity changes over time based on a differential equation. Instead of a single step, we calculate the "state" of the neuron repeatedly.

### Python Concept: While Loops
We use a `while` loop or a long `for` loop to simulate time passing.

In [ ]:
import matplotlib.pyplot as plt  # library for visualization

# Parameters for a simple rate model: tau * du/dt = -u + I
u = 0.0  # Initial membrane potential
input_current = 1.5
tau = 10.0  # Time constant
dt = 1.0  # Time step
time_steps = 100
activity_history = []

for t in range(time_steps):
  # The change in activity (Euler method)
  du = (-u + input_current) / tau * dt
  u += du
  activity_history.append(u)

# Plotting the results
plt.plot(activity_history)
plt.title("Rate-Based Neuron Activation Over Time")
plt.xlabel("Time Steps")
plt.ylabel("Firing Rate (u)")
plt.grid(True)
plt.show()

## 6. Solving the OR Gate (The Training Loop)
Now we will combine everything. We will create a **Dataset** (a list of lists) representing the OR gate and let the Perceptron "study" it over several **Epochs** (training rounds).

**The OR Gate Truth Table:**
| Input 1 | Input 2 | Target Output |
| :--- | :--- | :--- |
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

We will use a **Nested For Loop**:
1. The outer loop counts the **Epochs**.
2. The inner loop looks at each **Example** in our dataset.

In [ ]:
# Define the Dataset (Inputs and their correct Targets)
dataset = [
    ([0, 0], 0),
    ([0, 1], 1),
    ([1, 0], 1),
    ([1, 1], 1)
]

# Initialize Parameters
weights = [0.0, 0.0]  # Start with no knowledge
bias = 0.0
learning_rate = 0.1
epochs = 10  # How many times to repeat the training

print("Starting Training...")
print("-" * 30)


# The Training Loop
for epoch in range(epochs):
  total_error = 0

  for x_values, target in dataset:
    # Calculate Prediction (Weighted Sum + Bias)
    z = sum(x * w for x, w in zip(x_values, weights)) + bias
    prediction = 1 if z >= 0 else 0

    # Calculate Error
    error = target - prediction
    total_error += abs(error)

    # Update Weights and Bias if there's an error
    if error != 0:
      for i in range(len(weights)):
        weights[i] += learning_rate * error * x_values[i]
      bias += learning_rate * error

  print(f"Epoch {epoch+1}: Total Errors = {total_error}")

  # If no errors occurred, we can stop early!
  if total_error == 0:
    print("Training Complete! The neuron has learned the OR gate.")
    break

print("-" * 30)
print(f"Final Weights: {weights}")
print(f"Final Bias: {bias}")

## 7. Testing Your Trained Neuron
Now that the weights have been adjusted by the learning rule, we can test our neuron on any input to see if it behaves correctly.

### Python Concept: List Indexing
We can access specific parts of our results to verify the math.

In [ ]:
def test_neuron(test_input, weights, bias):
  z = sum(x * w for x, w in zip(test_input, weights)) + bias
  return 1 if z >= 0 else 0


# Let's test [0, 1]
test_val = [0, 1]
output = test_neuron(test_val, weights, bias)
print(f"Test Input: {test_val} -> Neuron Output: {output}")

if output == 1:
  print("Success! The neuron correctly identified an 'Active' state.")

## 8. Visualizing the Decision Boundary
In a 2D space, the Perceptron's weights and bias define a straight line. On one side of the line, the neuron outputs `0`; on the other, it outputs `1`.

The formula for this line is derived from:

$$w_1x_1 + w_2x_2 + b = 0$$

By rearranging this into the slope-intercept form ($y = mx + c$), we can plot it!

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_decision_boundary(weights, bias, dataset):
  # Plot the data points
  for x, target in dataset:
    color = 'red' if target == 1 else 'blue'
    marker = 'o' if target == 1 else 'x'
    plt.scatter(
        x[0], x[1], c=color,
        marker=marker, s=100,
        label=f"Class {target}"
    )

  # Calculate the boundary line
  # formula: w1*x + w2*y + b = 0  =>  y = -(w1/w2)x - (b/w2)
  x_coords = np.linspace(-0.5, 1.5, 10)

  if weights[1] != 0:
    y_coords = -(weights[0] / weights[1]) * x_coords - (bias / weights[1])
    plt.plot(
        x_coords, y_coords, color='black',
        linestyle='--', label='Decision Boundary'
    )

  # Formatting the plot
  plt.title("Perceptron Decision Boundary (OR Gate)")
  plt.xlabel("Input 1")
  plt.ylabel("Input 2")
  plt.xlim(-0.5, 1.5)
  plt.ylim(-0.5, 1.5)
  plt.axhline(0, color='black', linewidth=1)
  plt.axvline(0, color='black', linewidth=1)
  plt.grid(True, alpha=0.3)
  plt.show()

# Run the visualization using the weights from our training session
plot_decision_boundary(weights, bias, dataset)

## 9. Using a Real Dataset: Sklearn Iris
In professional coding, we don't always create datasets by hand. We use libraries like `sklearn` to load data.

We will use the **Iris flower dataset** (see below for more info). This contains measurements of different flowers. We will simplify it to a **Binary Classification** problem (Setosa vs. Versicolor) so our Perceptron can handle it.

### Python Concepts:
* **Imports**: Bringing in professional tools (`datasets`).
* **Slicing**: Taking only the parts of the list/array we need.


**Note**:
To test our Perceptron on real-world data, we will use the **Iris Flower Dataset**. This is one of the most famous datasets in history, introduced by the statistician Ronald Fisher in 1936.

It contains **150 samples** of flowers from three different species:
1.  **Iris Setosa**
2.  **Iris Versicolor**
3.  **Iris Virginica**

For each flower, four measurements (features) were taken in centimeters:
* **Sepal Length**
* **Sepal Width**
* **Petal Length**
* **Petal Width**

For this tutorial, we will simplify the problem to make it solvable by a single neuron (Perceptron).
* **Classes:** We will only look at two species: *Setosa* (Target 0) vs. *Versicolor* (Target 1).
* **Features:** We will only use two measurements: *Sepal Length* and *Sepal Width*.

You can view the full documentation and data source here:
[UCI Machine Learning Repository - Iris Dataset](https://archive.ics.uci.edu/ml/datasets/iris)

In [ ]:
from sklearn import datasets
import numpy as np
import matplotlib.pyplot as plt

# Load Data
iris = datasets.load_iris()
# We only take the first 100 samples (Setosa and Versicolor)
# We only take the first 2 features (Sepal Length, Sepal Width) for 2D plotting
n = 100
X = iris.data[:n, :2]
y = iris.target[:n]

# Standardization (Scaling the data)
# This makes the mean 0 and variance 1
X_std = np.copy(X)
X_std[:, 0] = (X[:, 0] - X[:, 0].mean()) / X[:, 0].std()
X_std[:, 1] = (X[:, 1] - X[:, 1].mean()) / X[:, 1].std()

# Train the Perceptron
weights = np.zeros(2)  # Start at 0
bias = 0.0
learning_rate = 0.01
epochs = 50

errors_history = []

print("Training Perceptron...")
for epoch in range(epochs):
  total_error = 0
  for i, target in enumerate(y):
    # Calculate weighted sum
    z = np.dot(X_std[i], weights) + bias

    # # If z is positive, predict 1. Otherwise, predict 0. (Activation)
    if z >= 0:
      prediction = 1
    else:
      prediction = 0

    error = target - prediction
    if error != 0:
      # Update weights and bias
      update = learning_rate * error
      weights += update * X_std[i]
      bias += update
      total_error += 1  # Count the mistake

    # Store the total number of errors for this epoch
    errors_history.append(total_error)

  if total_error == 0:
    print(f"Converged at epoch {epoch + 1}.")
    break


print(f"Final Weights: {weights}")
print(f"Final Bias: {bias}")

## 10. Final Visualization: Real World Boundary
Now we can see how our Perceptron separates the two types of Iris flowers based on their physical measurements!

In [ ]:
# Create the plot
plt.figure(figsize=(8, 6))

# Plot the Flower Data (Standardized)
plt.scatter(
    X_std[y == 0, 0], X_std[y == 0, 1],
    color='red', marker='o', label='Setosa'
)
plt.scatter(
    X_std[y == 1, 0], X_std[y == 1, 1],
    color='blue', marker='x', label='Versicolor'
)

# Calculate the Decision Boundary Line
# We want to find x2 (y-axis) for a given x1 (x-axis)
# w1*x1 + w2*x2 + b = 0  -->  x2 = -(w1*x1 + b) / w2
x_min, x_max = X_std[:, 0].min() - 1, X_std[:, 0].max() + 1
x_values = np.linspace(x_min, x_max, 100)
y_values = -(weights[0] * x_values + bias) / weights[1]

# Draw the line
plt.plot(x_values, y_values, label='Decision Boundary', color='green')

# Formatting
plt.xlabel('Sepal Length (Standardized)')
plt.ylabel('Sepal Width (Standardized)')
plt.legend(loc='upper left')
plt.title('Perceptron Classification: Setosa vs Versicolor')
plt.grid(True)
plt.show()